# 1. Data exploration
## Phase 3 data files
Logging in to Hugging Face, mounting Drive and listing the Phase 3 rhetorical role files in the dataset.


In [ ]:
import os
from huggingface_hub import login, get_token, list_repo_files
from google.colab import drive

login()
# Passing the token on so !python scripts can also download the data
os.environ["HF_TOKEN"] = get_token()

drive.mount('/content/drive')

# Listing every file in the dataset that belongs to Phase 3
all_files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
phase3_files = [f for f in all_files if "Phase_3" in f]

for f in phase3_files:
    print(f)


Mounted at /content/drive
Phase_3_RR/Test/CL_test.json
Phase_3_RR/Test/IT_test.json
Phase_3_RR/Train/CL_dev.json
Phase_3_RR/Train/CL_train.json
Phase_3_RR/Train/IT_dev.json
Phase_3_RR/Train/IT_train.json
Phase_3_RR/Train/label_vocab.json


## Phase 3 files and labels
Downloading the files and checking the labels and data format.


In [ ]:
# Downloading all the Phase 3 files and looking at how they are stored
import json
from huggingface_hub import hf_hub_download

paths = {}
for f in phase3_files:
    name = f.split("/")[-1]
    paths[name] = hf_hub_download("Exploration-Lab/CS-781-Capstone", f, repo_type="dataset")

with open(paths["label_vocab.json"]) as f:
    label_vocab = json.load(f)
print(label_vocab)

with open(paths["CL_train.json"]) as f:
    cl_train = json.load(f)

print(type(cl_train), len(cl_train))
print(str(cl_train)[:1500])


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

{'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}
<class 'list'> 40
[{'id': 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'text': ['Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.', 'The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers Association against glass bottle manufacturers in India alleging restrictive and monopolistic trade practices being adopted by the four major manufacturers namely: Haldyn Glass Gujarat Limited, Hindustan National Glass and Industries, AGI Glasspac and Mohan Crystal Glass Work who together control 90 of the g

## Documents, sentences and labels
Counting documents and sentences in each file and how often each label appears.


In [ ]:
# Loading all six data files
from collections import Counter

file_names = ["CL_train.json", "CL_dev.json", "CL_test.json",
              "IT_train.json", "IT_dev.json", "IT_test.json"]

data = {}
for name in file_names:
    with open(paths[name]) as f:
        data[name] = json.load(f)

# Looking at one test document
test_ids = list(data["CL_test.json"].keys())
first_test_doc = data["CL_test.json"][test_ids[0]]
print("CL test docs:", len(test_ids))
print("first test id:", test_ids[0])
print("first test doc:", type(first_test_doc))
print(first_test_doc)

# Counting documents, sentences and labels in train and dev
for name in ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json"]:
    docs = data[name]
    num_sentences = sum(len(doc["text"]) for doc in docs)
    label_counts = Counter(label for doc in docs for label in doc["labels"])

    print()
    print(name, "-", len(docs), "docs,", num_sentences, "sentences")
    print(label_counts.most_common())


CL test docs: 5
first test id: HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489
first test doc: <class 'list'>
['Manmohan, J. 1 .', 'Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th February, 2016 whereby the respondent-CCI has directed the opposite parties in Reference Case Number 8/2013 to file their suggestions objections to the report of Director General (for short DG) and furnish their audited balance sheet, profit and loss account, turnover for the last three financial years.', 'The CCI has further directed individuals who have been identified in the DGs investigation report as key persons of the parties to appear before the CCI for oral hearing on the investigation report of DG.', 'It is pertinent to mention that the DG had found five leading tyre manufacturing companies under aegis of their association being Automotive Tyre Manufacturers Association (ATMA) who have 

## Sentence lengths in tokens
Checking sentence and document lengths with the Phase-1 tokenizer.


In [ ]:
# Counting tokens per sentence and per document with the Phase-1 tokenizer
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")

print("IT test docs:", len(data["IT_test.json"]))

for name in ["CL_train.json", "IT_train.json"]:
    sentence_lengths = []
    doc_lengths = []
    for doc in data[name]:
        lengths = [len(tokenizer.encode(s).ids) for s in doc["text"]]
        sentence_lengths.extend(lengths)
        doc_lengths.append(sum(lengths))

    sentence_lengths.sort()
    print()
    print(name)
    print("tokens per sentence - mean:", sum(sentence_lengths) // len(sentence_lengths),
          "| 95%:", sentence_lengths[int(0.95 * len(sentence_lengths))],
          "| max:", sentence_lengths[-1])
    print("tokens per doc - mean:", sum(doc_lengths) // len(doc_lengths),
          "| max:", max(doc_lengths))


IT test docs: 5

CL_train.json
tokens per sentence - mean: 39 | 95%: 86 | max: 865
tokens per doc - mean: 10480 | max: 71975

IT_train.json
tokens per sentence - mean: 42 | 95%: 102 | max: 1187
tokens per doc - mean: 6687 | max: 19832


# 2. Data preparation
## Building the windows
Running data_prep.py with document context and with one sentence per window.


In [ ]:
import os
from huggingface_hub import login, get_token
from google.colab import drive

login()
os.environ["HF_TOKEN"] = get_token()
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase3/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p3 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json

!python /content/drive/MyDrive/CS781/phase3/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p3_sentence \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json \
    --context sentence


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Building windows (document context) ...
  CL: 40 documents, 1732 windows
  IT: 40 documents, 1076 windows
  CL: 5 documents, 232 windows
  IT: 5 documents, 117 windows
  CL: 5 documents, 213 windows
  IT: 5 documents, 155 windows

Saved to /content/drive/MyDrive/CS781/data_p3
Train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
Labels: ['Fact', 'Issue', 'Argument

## Testing the model
Running the model.py self-test with two dummy windows.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/model.py


RR model parameters: 111.637773 M
logits shape: torch.Size([2, 3, 13])
loss: 2.5801544189453125


## Testing methods.py
Checking full, LoRA, LoRMA, IA3 and AdaLoRA and how many weights each one trains.


In [ ]:
# Building a small model with each method and counting trainable weights
import sys
sys.path.append("/content/drive/MyDrive/CS781/phase3")

from model import ModelConfig, build_rr_model
from methods import setup_method, count_trainable

config = ModelConfig(arch="convmix", n_layer=3)
for method in ["full", "lora", "lorma", "ia3", "adalora"]:
    model = build_rr_model(config, num_labels=13)
    setup_method(model, method, total_steps=100)
    trainable, total = count_trainable(model)
    print(method, "trainable:", trainable, "of", total)


full trainable: 46755597 of 46755597
lora trainable: 2144781 of 46792461
lorma trainable: 2157069 of 46804749
ia3 trainable: 2116365 of 46764045
adalora trainable: 2255445 of 46903131


# 3. Training

## Full fine-tuning: decoder

Training all the weights of the Phase-1 decoder along with the new
sentence head, on the CL and IT train data. The best model on the
dev set is saved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase3/train.py \
    --arch decoder --method full \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_full


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
2808 train windows, 349 dev windows
702 steps per epoch, 2106 steps total
Class weights: [0.56, 3.48, 0.9, 0.98, 1.09, 2.14, 0.82, 1.51, 10.0, 1.22, 0.55, 2.05, 2.09]
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 111.64M of 111.6M parameters (100.00%)
step 20/2106  epoch 1  loss 2.3841  elapsed 0.1 min
step 40/2106  epoch 1  loss 2.1099  elapsed 0.1 min
step 60/2106  epoch 1  loss 1.8550  elapsed 0.2 min
step 80/2106  epoch 1  loss 1.6252  elapsed 0.2 min
step 100/2106  epoch 1  loss 1.5575  elapsed 0.3 min
step 120/2106  epoch 1  loss 1.7017  elapsed 0.4 min
step 140/2106  epoch 1  loss 1.4115  elapsed 0.4 min
step 160/2106  epoch 1  loss 1.5588  elapsed 0.5 min
step 180/2106  epoch 1  loss 1.6320  elapsed 0.5 min
step 200/

# 4. Prediction and submissions

## Submission 1: decoder full fine-tuning

Checking the scores for each label on the dev set, then predicting the
test documents and making the Codabench submission file.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_full

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/decoder_full


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/decoder_full/best_checkpoint.pt (step 1600, dev macro-F1 0.5100)
Dev macro-F1: 0.5100
  CL: 0.4178
  IT: 0.5398
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.58      0.40      0.48       400
    ArgumentRespondent       0.63      0.37      0.47       272
                  Fact       0.68      0.77      0.72       444
                 Issue       0.50      0.53      0.52        15
                  None       0.32      0.24      0.28        33
PrecedentNotReliedUpon       0.28      0.40      0.33        50
   PrecedentReliedUpon       0.48      0.50      0.49       254
    RatioOfTheDecision       0.52      0.76      0.62       403
    RulingByLowerCourt       0.52      0.36      0.43       102
  RulingByPresentCourt       0.55      0.43      0.48        49
               Statute       0.78      0.83      0.80       120

              accuracy                           0.58  

## LoRMA: decoder

Freezing the Phase-1 decoder and training only the LoRMA matrices
(rank 8) on the attention layers and the sentence head.



In [ ]:
!python /content/drive/MyDrive/CS781/phase3/train.py \
    --arch decoder --method lorma \
    --epochs 5 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_lorma


Using device: cuda
2808 train windows, 349 dev windows
702 steps per epoch, 3510 steps total
Class weights: [0.56, 3.48, 0.9, 0.98, 1.09, 2.14, 0.82, 1.51, 10.0, 1.22, 0.55, 2.05, 2.09]
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: lorma, training 2.70M of 112.2M parameters (2.40%)
step 20/3510  epoch 1  loss 2.3897  elapsed 0.1 min
step 40/3510  epoch 1  loss 2.1459  elapsed 0.1 min
step 60/3510  epoch 1  loss 1.9473  elapsed 0.1 min
step 80/3510  epoch 1  loss 1.7878  elapsed 0.2 min
step 100/3510  epoch 1  loss 1.7667  elapsed 0.2 min
step 120/3510  epoch 1  loss 1.8538  elapsed 0.2 min
step 140/3510  epoch 1  loss 1.6509  elapsed 0.3 min
step 160/3510  epoch 1  loss 1.8246  elapsed 0.3 min
step 180/3510  epoch 1  loss 1.7194  elapsed 0.3 min
step 200/3510  epoch 1  loss 1.5568  elapsed 0.4 min
  dev macro-F1: 0.3224  (CL 0.2961, IT 0.3437)
step 220/3510  epoch 1  loss 1.4083  elap

## Submission 2: decoder LoRMA

Checking the scores for each label on the dev set, then predicting the
test documents and making the Codabench submission file.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_lorma

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/decoder_lorma


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/decoder_lorma/best_checkpoint.pt (step 2800, dev macro-F1 0.5319)
Dev macro-F1: 0.5319
  CL: 0.4563
  IT: 0.5378
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.63      0.41      0.50       400
    ArgumentRespondent       0.61      0.37      0.46       272
                  Fact       0.65      0.67      0.66       444
                 Issue       0.71      0.67      0.69        15
                  None       0.35      0.24      0.29        33
PrecedentNotReliedUpon       0.27      0.32      0.29        50
   PrecedentReliedUpon       0.54      0.45      0.49       254
    RatioOfTheDecision       0.47      0.89      0.62       403
    RulingByLowerCourt       0.72      0.35      0.47       102
  RulingByPresentCourt       0.59      0.53      0.56        49
               Statute       0.87      0.77      0.81       120

              accuracy                           0.57 

## LoRA: encdec

Freezing the Phase-1 encdec encoder and training only the LoRA matrices
(rank 8) on the attention layers and the sentence head.



In [ ]:
!python /content/drive/MyDrive/CS781/phase3/train.py \
    --arch encdec --method lora \
    --epochs 5 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/encdec_lora


Using device: cuda
2808 train windows, 349 dev windows
702 steps per epoch, 3510 steps total
Class weights: [0.56, 3.48, 0.9, 0.98, 1.09, 2.14, 0.82, 1.51, 10.0, 1.22, 0.55, 2.05, 2.09]
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 68
Method: lora, training 2.18M of 60.9M parameters (3.58%)
step 20/3510  epoch 1  loss 2.4168  elapsed 0.0 min
step 40/3510  epoch 1  loss 2.3090  elapsed 0.1 min
step 60/3510  epoch 1  loss 2.3327  elapsed 0.1 min
step 80/3510  epoch 1  loss 2.2600  elapsed 0.1 min
step 100/3510  epoch 1  loss 2.2346  elapsed 0.1 min
step 120/3510  epoch 1  loss 2.2530  elapsed 0.1 min
step 140/3510  epoch 1  loss 2.2781  elapsed 0.1 min
step 160/3510  epoch 1  loss 2.2370  elapsed 0.2 min
step 180/3510  epoch 1  loss 2.2169  elapsed 0.2 min
step 200/3510  epoch 1  loss 2.1755  elapsed 0.2 min
  dev macro-F1: 0.1088  (CL 0.1072, IT 0.1304)
step 220/3510  epoch 1  loss 2.0932  elapse

## Submission 3: encdec LoRA

Checking the scores for each label on the dev set, then predicting the
test documents and making the Codabench submission file.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/encdec_lora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/encdec_lora

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/encdec_lora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/encdec_lora


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/encdec_lora/best_checkpoint.pt (step 3200, dev macro-F1 0.2644)
Dev macro-F1: 0.2638
  CL: 0.2591
  IT: 0.2780
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.38      0.35      0.36       400
    ArgumentRespondent       0.33      0.04      0.07       272
                  Fact       0.53      0.59      0.56       444
                 Issue       0.00      0.00      0.00        15
                  None       0.00      0.00      0.00        33
PrecedentNotReliedUpon       0.00      0.00      0.00        50
   PrecedentReliedUpon       0.29      0.41      0.34       254
    RatioOfTheDecision       0.35      0.55      0.43       403
    RulingByLowerCourt       0.31      0.39      0.34       102
  RulingByPresentCourt       1.00      0.04      0.08        49
               Statute       0.75      0.71      0.73       120

              accuracy                           0.40   

## IA3: moe

Keeping the Phase-1 moe model frozen and training only small IA3
scaling vectors along with the sentence head.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase3/train.py \
    --arch moe --method ia3 \
    --epochs 5 --learning_rate 3e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/moe_ia3


Mounted at /content/drive
Using device: cuda
2808 train windows, 349 dev windows
702 steps per epoch, 3510 steps total
Class weights: [0.56, 3.48, 0.9, 0.98, 1.09, 2.14, 0.82, 1.51, 10.0, 1.22, 0.55, 2.05, 2.09]
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: ia3, training 1.95M of 196.2M parameters (0.99%)
step 20/3510  epoch 1  loss 2.4184  elapsed 0.1 min
step 40/3510  epoch 1  loss 2.2193  elapsed 0.1 min
step 60/3510  epoch 1  loss 2.0699  elapsed 0.2 min
step 80/3510  epoch 1  loss 2.0184  elapsed 0.3 min
step 100/3510  epoch 1  loss 1.9037  elapsed 0.3 min
step 120/3510  epoch 1  loss 1.9953  elapsed 0.4 min
step 140/3510  epoch 1  loss 1.8414  elapsed 0.4 min
step 160/3510  epoch 1  loss 1.8903  elapsed 0.5 min
step 180/3510  epoch 1  loss 1.9743  elapsed 0.5 min
step 200/3510  epoch 1  loss 1.7699  elapsed 0.6 min
  dev macro-F1: 0.2528  (CL 0.2343, IT 0.2726)
step 220/3510  epoch 

## Submission 4: moe IA3

Checking the scores for each label on the dev set, then predicting the
test documents and making the Codabench submission file.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/moe_ia3/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/moe_ia3

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/moe_ia3/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/moe_ia3


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/moe_ia3/best_checkpoint.pt (step 3510, dev macro-F1 0.4527)
Dev macro-F1: 0.4527
  CL: 0.4031
  IT: 0.4465
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.55      0.39      0.46       400
    ArgumentRespondent       0.32      0.13      0.18       272
                  Fact       0.61      0.69      0.65       444
                 Issue       0.58      0.47      0.52        15
                  None       0.36      0.27      0.31        33
PrecedentNotReliedUpon       0.31      0.64      0.42        50
   PrecedentReliedUpon       0.46      0.37      0.41       254
    RatioOfTheDecision       0.41      0.67      0.51       403
    RulingByLowerCourt       0.23      0.20      0.21       102
  RulingByPresentCourt       0.53      0.53      0.53        49
               Statute       0.79      0.78      0.79       120

              accuracy                           0.49      2

## AdaLoRA: convmix

Keeping the Phase-1 convmix model frozen and training AdaLoRA, a LoRA
version that changes the rank of each layer during training, along
with the sentence head.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/train.py \
    --arch convmix --method adalora \
    --epochs 5 --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/convmix_adalora


Using device: cuda
2808 train windows, 349 dev windows
702 steps per epoch, 3510 steps total
Class weights: [0.56, 3.48, 0.9, 0.98, 1.09, 2.14, 0.82, 1.51, 10.0, 1.22, 0.55, 2.05, 2.09]
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: adalora, training 2.70M of 107.6M parameters (2.51%)
step 20/3510  epoch 1  loss 2.4068  elapsed 0.1 min
step 40/3510  epoch 1  loss 2.2130  elapsed 0.1 min
step 60/3510  epoch 1  loss 2.0924  elapsed 0.2 min
step 80/3510  epoch 1  loss 2.0378  elapsed 0.2 min
step 100/3510  epoch 1  loss 1.9642  elapsed 0.2 min
step 120/3510  epoch 1  loss 2.0189  elapsed 0.3 min
step 140/3510  epoch 1  loss 1.8896  elapsed 0.3 min
step 160/3510  epoch 1  loss 1.8774  elapsed 0.3 min
step 180/3510  epoch 1  loss 1.9984  elapsed 0.4 min
step 200/3510  epoch 1  loss 1.7773  elapsed 0.4 min
  dev macro-F1: 0.2725  (CL 0.2272, IT 0.3138)
step 220/3510  epoch 1  loss 1.6551  el

## Submission 5: convmix AdaLoRA

Checking the convmix AdaLoRA model on the dev set and making its test submission.



In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/convmix_adalora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/convmix_adalora

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/convmix_adalora/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/convmix_adalora


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/convmix_adalora/best_checkpoint.pt (step 3510, dev macro-F1 0.5130)
Dev macro-F1: 0.5128
  CL: 0.4477
  IT: 0.5056
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.61      0.41      0.49       400
    ArgumentRespondent       0.54      0.31      0.40       272
                  Fact       0.62      0.70      0.66       444
                 Issue       0.78      0.47      0.58        15
                  None       0.30      0.24      0.27        33
PrecedentNotReliedUpon       0.39      0.60      0.47        50
   PrecedentReliedUpon       0.55      0.47      0.51       254
    RatioOfTheDecision       0.46      0.77      0.57       403
    RulingByLowerCourt       0.56      0.27      0.37       102
  RulingByPresentCourt       0.55      0.49      0.52        49
               Statute       0.85      0.78      0.81       120

              accuracy                           0.5

## Knowledge distillation: decoder

Training a smaller 6-layer decoder, taken from the first 6 layers of the
Phase-1 decoder, to learn from the trained decoder_full model and from
the gold labels.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_full/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_distill


Using device: cuda
Student backbone loaded, missing keys: 0, unused keys: 43
Teacher: 111.6M parameters, student: 69.2M parameters (62% of the teacher)
702 steps per epoch, 2106 steps total
step 20/2106  epoch 1  loss 4.4925  elapsed 0.1 min
step 40/2106  epoch 1  loss 3.9304  elapsed 0.1 min
step 60/2106  epoch 1  loss 3.7866  elapsed 0.1 min
step 80/2106  epoch 1  loss 3.4418  elapsed 0.2 min
step 100/2106  epoch 1  loss 3.1017  elapsed 0.2 min
step 120/2106  epoch 1  loss 3.5362  elapsed 0.3 min
step 140/2106  epoch 1  loss 2.8403  elapsed 0.3 min
step 160/2106  epoch 1  loss 3.0815  elapsed 0.3 min
step 180/2106  epoch 1  loss 2.8680  elapsed 0.4 min
step 200/2106  epoch 1  loss 2.8661  elapsed 0.4 min
  dev macro-F1: 0.2825  (CL 0.2802, IT 0.3085)
step 220/2106  epoch 1  loss 2.5523  elapsed 0.5 min
step 240/2106  epoch 1  loss 2.8109  elapsed 0.6 min
step 260/2106  epoch 1  loss 2.6908  elapsed 0.6 min
step 280/2106  epoch 1  loss 2.8110  elapsed 0.6 min
step 300/2106  epoch 1  l

## Submission 6: decoder distillation

Checking the 6-layer student on the dev set and making its test submission.


In [ ]:
!python /content/drive/MyDrive/CS781/phase3/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p3/decoder_distill

!python /content/drive/MyDrive/CS781/phase3/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p3/decoder_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p3 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p3/decoder_distill


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p3/decoder_distill/best_checkpoint.pt (step 1800, dev macro-F1 0.4658)
Dev macro-F1: 0.4658
  CL: 0.3891
  IT: 0.5097
                        precision    recall  f1-score   support

    ArgumentPetitioner       0.50      0.47      0.48       400
    ArgumentRespondent       0.38      0.14      0.20       272
                  Fact       0.64      0.68      0.66       444
                 Issue       0.53      0.53      0.53        15
                  None       0.41      0.27      0.33        33
PrecedentNotReliedUpon       0.18      0.52      0.26        50
   PrecedentReliedUpon       0.44      0.42      0.43       254
    RatioOfTheDecision       0.47      0.63      0.54       403
    RulingByLowerCourt       0.42      0.31      0.36       102
  RulingByPresentCourt       0.62      0.49      0.55        49
               Statute       0.81      0.76      0.78       120

              accuracy                           0.5